# Data Preprocessing

## Objective

The objective of this notebook is to prepare the dataset for exploratory analysis and machine learning by addressing the data quality issues identified in the previous stage.

The preprocessing pipeline includes handling invalid values, treating missing values, addressing anomalous observations, managing extreme outliers, and validating the cleaned dataset before model development.

The resulting cleaned dataset will be used for Business EDA, Feature Engineering, and Model Training.

In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)

In [4]:
credit_df = pd.read_csv("../data/raw/GiveMeSomeCredit-training.csv")

In [5]:
credit_clean = credit_df.copy()

# Preprocessing Plan

Based on the Data Quality Assessment, the following preprocessing steps will be performed to prepare the dataset for exploratory analysis and machine learning.

| Feature | Issue Identified | Planned Treatment |
|---------|------------------|-------------------|
| Unnamed: 0 | Non-informative identifier | Drop the column |
| Age | One invalid value (0) | Remove the invalid record |
| Delinquency Features | Anomalous values (96 and 98) | Replace with NaN and impute using the mode |
| MonthlyIncome | 19.82% missing values | Impute using the median |
| NumberOfDependents | 2.62% missing values | Impute using the median |
| RevolvingUtilizationOfUnsecuredLines | Extreme outliers | Evaluate and apply an appropriate outlier treatment if necessary |
| DebtRatio | Extreme outliers and highly skewed distribution | Evaluate and apply an appropriate outlier treatment if necessary |
| NumberOfOpenCreditLinesAndLoans | No data quality issues | No preprocessing required |
| NumberRealEstateLoansOrLines | No data quality issues | No preprocessing required |
| Duplicate Records | No duplicate records detected | No action required |

## Preprocessing Workflow

The preprocessing steps will be performed in the following order:

1. Remove non-informative columns.
2. Remove invalid records.
3. Handle anomalous values.
4. Treat missing values.
5. Handle extreme outliers where appropriate.
6. Validate the cleaned dataset.
7. Save the final preprocessed dataset for subsequent analysis and model development.

Each preprocessing step will be validated before proceeding to the next stage to ensure data integrity and reproducibility.


## Problem

During the Data Quality Assessment, one record was found with an age value of **0**, which is not a valid age for a loan applicant.

## Treatment

Since this represents only **one record (0.00067% of the dataset)**, it is removed from the dataset rather than being imputed.

This ensures that only valid customer records are retained for subsequent analysis.

In [6]:
# Number of rows before removing invalid age
print(f"Rows before removal: {credit_clean.shape[0]}")

# Number of invalid age records
invalid_age = (credit_clean["age"] == 0).sum()
print(f"Invalid age records: {invalid_age}")

# Remove invalid age records
credit_clean = credit_clean[credit_clean["age"] != 0].copy()

# Verify
print(f"Rows after removal: {credit_clean.shape[0]}")

# Final check
print(f"Remaining invalid age records: {(credit_clean['age'] == 0).sum()}")

Rows before removal: 150000
Invalid age records: 1
Rows after removal: 149999
Remaining invalid age records: 0


### Result

- Successfully removed the single record with an invalid age value.
- The dataset now contains **149,999** valid customer records.
- No remaining records have an age value of **0**.

# 2. Handle Anomalous Delinquency Values

## Problem

During the Data Quality Assessment, the delinquency-related features were found to contain anomalous values of **96** and **98**. These values are unlikely to represent the actual number of late payment occurrences and appear to be placeholder or coded values.

## Treatment

The anomalous values are replaced with **missing values (NaN)** rather than being removed or arbitrarily modified.

This preserves all customer records while allowing the missing values to be handled consistently during the imputation stage.

In [7]:
# Delinquency columns
delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

# Count anomalous values before replacement
for col in delinquency_cols:
    count = credit_clean[col].isin([96, 98]).sum()
    print(f"{col}: {count} anomalous values")

# Replace 96 and 98 with NaN
credit_clean[delinquency_cols] = credit_clean[delinquency_cols].replace([96, 98], np.nan)

# Verify replacement
print("\nAfter replacement:\n")

for col in delinquency_cols:
    print(f"{col}")
    print(credit_clean[col].isna().sum())
    print("-" * 50)

NumberOfTime30-59DaysPastDueNotWorse: 269 anomalous values
NumberOfTime60-89DaysPastDueNotWorse: 269 anomalous values
NumberOfTimes90DaysLate: 269 anomalous values

After replacement:

NumberOfTime30-59DaysPastDueNotWorse
269
--------------------------------------------------
NumberOfTime60-89DaysPastDueNotWorse
269
--------------------------------------------------
NumberOfTimes90DaysLate
269
--------------------------------------------------


# Step 3: Missing Value Treatment

## Objective

The dataset contains missing values in several features. Since the nature and distribution of these features differ, a single imputation strategy is not appropriate for all variables.

The following strategies are adopted:

| Feature | Imputation Strategy | Justification |
|---------|---------------------|---------------|
| MonthlyIncome | Median | Highly right-skewed with extreme outliers |
| NumberOfDependents | Median | Numerical count feature with low missing percentage |
| Delinquency Features | Mode | Count-based variables where zero is the most frequent value |

In [8]:
missing_before = credit_clean.isnull().sum()

missing_before[missing_before > 0].sort_values(ascending=False)

MonthlyIncome                           29731
NumberOfDependents                       3924
NumberOfTime30-59DaysPastDueNotWorse      269
NumberOfTimes90DaysLate                   269
NumberOfTime60-89DaysPastDueNotWorse      269
dtype: int64

In [9]:
# Monthly Income
credit_clean["MonthlyIncome"] = credit_clean["MonthlyIncome"].fillna(
    credit_clean["MonthlyIncome"].median()
)

# Number of Dependents
credit_clean["NumberOfDependents"] = credit_clean["NumberOfDependents"].fillna(
    credit_clean["NumberOfDependents"].median()
)

In [12]:
delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

for col in delinquency_cols:
    credit_clean[col] = credit_clean[col].fillna(
        credit_clean[col].mode()[0]
    )

In [13]:
missing_after = credit_clean.isnull().sum()

missing_after[missing_after > 0]

Series([], dtype: int64)

In [15]:
check_point_1= credit_clean
check_point_1

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2.0,0.802982,9120.0,13,0.0,6,0.0,2.0
1,2,0,0.957151,40,0.0,0.121876,2600.0,4,0.0,0,0.0,1.0
2,3,0,0.658180,38,1.0,0.085113,3042.0,2,1.0,0,0.0,0.0
3,4,0,0.233810,30,0.0,0.036050,3300.0,5,0.0,0,0.0,0.0
4,5,0,0.907239,49,1.0,0.024926,63588.0,7,0.0,1,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
149995,149996,0,0.040674,74,0.0,0.225131,2100.0,4,0.0,1,0.0,0.0
149996,149997,0,0.299745,44,0.0,0.716562,5584.0,4,0.0,1,0.0,2.0
149997,149998,0,0.246044,58,0.0,3870.000000,5400.0,18,0.0,1,0.0,0.0
149998,149999,0,0.000000,30,0.0,0.000000,5716.0,4,0.0,0,0.0,0.0


In [16]:
# Check if the column exists
if "Unnamed: 0" in credit_clean.columns:
    credit_clean.drop(columns=["Unnamed: 0"], inplace=True)

# Verify
credit_clean.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2.0,0.802982,9120.0,13,0.0,6,0.0,2.0
1,0,0.957151,40,0.0,0.121876,2600.0,4,0.0,0,0.0,1.0
2,0,0.658180,38,1.0,0.085113,3042.0,2,1.0,0,0.0,0.0
3,0,0.233810,30,0.0,0.036050,3300.0,5,0.0,0,0.0,0.0
4,0,0.907239,49,1.0,0.024926,63588.0,7,0.0,1,0.0,0.0


# Step 6: Final Data Validation

## Objective

Before proceeding to Business EDA and model development, the cleaned dataset is validated to ensure that all preprocessing steps have been applied correctly.

The validation checks include:

- Dataset dimensions
- Data types
- Missing values
- Duplicate records
- Invalid values
- Summary statistics

This step ensures that the dataset is consistent, complete, and ready for downstream analysis.

In [17]:
print("Dataset Shape:", credit_clean.shape)

Dataset Shape: (149999, 11)


In [18]:
credit_clean.info()

<class 'pandas.DataFrame'>
Index: 149999 entries, 0 to 149999
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      149999 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  149999 non-null  float64
 2   age                                   149999 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  149999 non-null  float64
 4   DebtRatio                             149999 non-null  float64
 5   MonthlyIncome                         149999 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       149999 non-null  int64  
 7   NumberOfTimes90DaysLate               149999 non-null  float64
 8   NumberRealEstateLoansOrLines          149999 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  149999 non-null  float64
 10  NumberOfDependents                    149999 non-null  float64
dtypes: float64(7), i

In [19]:
missing = credit_clean.isnull().sum()

missing[missing > 0]

Series([], dtype: int64)

In [20]:
duplicates = credit_clean.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 796


In [21]:
duplicates = credit_clean[credit_clean.duplicated(keep=False)]

duplicates.sort_values(list(credit_clean.columns)).head(20)

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
134365,0,0.0,21,0.0,0.0,0.0,1,0.0,0,0.0,0.0
137102,0,0.0,21,0.0,0.0,0.0,1,0.0,0,0.0,0.0
25632,0,0.0,21,0.0,0.0,820.0,2,0.0,0,0.0,0.0
83552,0,0.0,21,0.0,0.0,820.0,2,0.0,0,0.0,0.0
139345,0,0.0,21,0.0,0.0,820.0,2,0.0,0,0.0,0.0
60376,0,0.0,21,0.0,0.0,5400.0,1,0.0,0,0.0,0.0
100594,0,0.0,21,0.0,0.0,5400.0,1,0.0,0,0.0,0.0
127724,0,0.0,21,0.0,0.0,5400.0,1,0.0,0,0.0,0.0
44006,0,0.0,22,0.0,0.0,0.0,1,0.0,0,0.0,0.0
54653,0,0.0,22,0.0,0.0,0.0,1,0.0,0,0.0,0.0


Duplicate Check

A total of 796 duplicate rows were identified.

However, since the dataset does not contain a unique customer identifier, it is not possible to determine whether these rows represent true duplicate records or different customers with identical feature values.

Therefore, no duplicate rows were removed to avoid potential loss of valid customer observations.

In [24]:
print(
    "Age = 0 Records:",
    (credit_clean["age"] == 0).sum()
)

Age = 0 Records: 0


In [25]:
delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

for col in delinquency_cols:
    print(f"{col}:")
    print("Missing :", credit_clean[col].isnull().sum())
    print("96 Count:", (credit_clean[col] == 96).sum())
    print("98 Count:", (credit_clean[col] == 98).sum())
    print("-" * 40)

NumberOfTime30-59DaysPastDueNotWorse:
Missing : 0
96 Count: 0
98 Count: 0
----------------------------------------
NumberOfTime60-89DaysPastDueNotWorse:
Missing : 0
96 Count: 0
98 Count: 0
----------------------------------------
NumberOfTimes90DaysLate:
Missing : 0
96 Count: 0
98 Count: 0
----------------------------------------


In [26]:
credit_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
SeriousDlqin2yrs,149999.0,0.066840,0.249746,0.0,0.000000,0.000000,0.000000,1.0
RevolvingUtilizationOfUnsecuredLines,149999.0,6.048472,249.756203,0.0,0.029867,0.154176,0.559044,50708.0
age,149999.0,52.295555,14.771298,21.0,41.000000,52.000000,63.000000,109.0
NumberOfTime30-59DaysPastDueNotWorse,149999.0,0.245348,0.697231,0.0,0.000000,0.000000,0.000000,13.0
DebtRatio,149999.0,353.007426,2037.825113,0.0,0.175074,0.366503,0.868257,329664.0
MonthlyIncome,149999.0,6418.457710,12890.438465,0.0,3903.000000,5400.000000,7400.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,149999.0,8.452776,5.145964,0.0,5.000000,8.000000,11.000000,58.0
NumberOfTimes90DaysLate,149999.0,0.090294,0.485108,0.0,0.000000,0.000000,0.000000,17.0
NumberRealEstateLoansOrLines,149999.0,1.018233,1.129772,0.0,0.000000,1.000000,2.000000,54.0
NumberOfTime60-89DaysPastDueNotWorse,149999.0,0.064707,0.329790,0.0,0.000000,0.000000,0.000000,11.0


# Preprocessing Summary

The preprocessing pipeline was completed based on the findings from the Data Quality Assessment. The primary objective was to improve data quality while preserving valid customer information.

The following preprocessing steps were performed:

- Removed the non-informative `Unnamed: 0` column.
- Removed the single record with an invalid age (`age = 0`).
- Replaced anomalous delinquency values (`96` and `98`) with missing values and imputed them using the mode.
- Imputed missing values in `MonthlyIncome` using the median.
- Imputed missing values in `NumberOfDependents` using the median.
- Preserved extreme but plausible observations in `DebtRatio` and `RevolvingUtilizationOfUnsecuredLines` for further evaluation during feature engineering and model development.
- Retained duplicate records because the dataset does not contain a unique customer identifier, making it impossible to distinguish true duplicate records from different customers with identical attributes.

The cleaned dataset was validated to ensure that the preprocessing steps were applied correctly and is now ready for exploratory analysis.

In [27]:
credit_clean.to_csv("../data/processed/credit_cleaned.csv", index=False)